# True Crime Pipeline — Colab GPU Runner

Edit code in **Cursor**, push to GitHub, then run the **Sync & test run** cell below.

1. **Runtime → Change runtime type → GPU** (A100 if your Colab plan provides it)
2. Run **Setup** cells once per Colab session
3. After each `git push` from Cursor, run **Sync & test run** only

## Setup (once per Colab session)

In [ ]:
import os

REPO_URL = "https://github.com/Paarth-Rana/Agent-Based-TrueCrime-Content-Generation-Pipeline.git"
REPO_DIR = "/content/Agent-Based-TrueCrime-Content-Generation-Pipeline"
BRANCH = "main"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL}
else:
    %cd {REPO_DIR}
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q -r requirements.txt
print("Repo ready:", REPO_DIR)

In [ ]:
# Optional: persist Hugging Face downloads across Colab disconnects
USE_DRIVE_CACHE = True  # set False to skip

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)
    print("HF_HOME:", os.environ["HF_HOME"])
else:
    print("Using default Colab HF cache (cleared when runtime disconnects)")

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: no GPU — pipeline will be very slow on CPU.")

In [ ]:
# Loads Qwen + Bark + SDXL (several minutes first time; uses HF cache if set)
%cd /content/Agent-Based-TrueCrime-Content-Generation-Pipeline

import pipeline
print("Models loaded. Ready for test runs.")

## Sync & test run (after each Cursor edit + `git push`)

Do **not** restart runtime for normal code changes. Only re-run setup if you changed `requirements.txt` or hit OOM.

In [ ]:
import importlib

TOPIC = "D. B. Cooper"  # or "" for automatic topic selection
BRANCH = "main"

%cd /content/Agent-Based-TrueCrime-Content-Generation-Pipeline
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

import pipeline
importlib.reload(pipeline)

state = pipeline.run_pipeline(TOPIC)
print("Chosen:", state.get("source_title"))
print("Wiki:", state.get("source_url"))
print("Output folder:", state.get("out_dir"))
print("Manifest:", state.get("manifest_path"))

## Download outputs (optional)

In [ ]:
from google.colab import files

!zip -r /content/truecrime_outputs.zip outputs/
files.download("/content/truecrime_outputs.zip")